# 📌 **Requirements Part:**

In [ ]:
# 📦 Standard Library
import argparse
import csv
import json
import math
import numbers
import os
import pickle
import random
import sys
import time
from tqdm import tqdm
from collections import OrderedDict

In [ ]:
# 🧪 Scientific and Data Handling Libraries
import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
# 🧬 Chemistry Libraries
from pubchempy import get_compounds, download
from rdkit import Chem

In [ ]:
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.warning')

In [ ]:
# 🌐 Graph and Network Libraries
import networkx as nx

In [ ]:
# 🔥 PyTorch & PyTorch Geometric
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data, InMemoryDataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool
from torch_geometric.nn import global_max_pool  # ← ممکنه استفاده بشه

In [ ]:
# Local experiment directory (launch Jupyter from this directory)
from pathlib import Path
folder = str(Path.cwd().resolve()) + "/"

---
---
---

# 📌 **Preprocessing Part:**

## 1️⃣ **def load_drug_list():**

In [ ]:
def load_drug_list():
    '''
      Returns list of all drugs in DrugList.csv
    '''
    filename = folder + "molecular_features.csv"

    with open(filename, "rt") as csvfile:
        reader = csv.reader(csvfile)
        next(reader, None)  # رد کردن header
        drugs = []
        for line in reader:
            drugs.append(line[0]) # line[5]
    drugs = list(set(drugs))  # حذف موارد تکراری
    return drugs

### 🔍 Test Time!

In [ ]:
drug_list = load_drug_list()
print(drug_list)

In [ ]:
print(f"تعداد کل: {len(drug_list)}, تکراری دارد: {'نه' if len(drug_list) == len(set(drug_list)) else 'بله'}")

---
---

## 2️⃣ **def write_drug_cid():**

---
---

## 3️⃣ **def is_not_float():**

---
---

## 4️⃣ **def cid_from_other_source():**

---
---

## 5️⃣ **def load_cid_dict():**

---
---

## 6️⃣ **def download_smiles():**

---
---
---

# 📌 **SMILES into One-Hot Part:** (Still "Preprocessing Part!")

*The following codes will convert the SMILES format into one-hot format.*

## 7️⃣ **def one_of_k_encoding():**

In [ ]:
def one_of_k_encoding(x, allowable_set):
    """
    One-hot encode the input x based on the allowable_set.

    Parameters:
    - x: input value (int, str, etc.)
    - allowable_set: list of allowed values to encode

    Returns:
    - List of 0/1 values with 1 only at the index where x matches
    """
    if x not in allowable_set:
        raise ValueError(f"Value '{x}' not in allowable set: {allowable_set}")
    return [int(x == s) for s in allowable_set]

### 🔍 Test Time!

In [ ]:
test_result = one_of_k_encoding('N', ['C', 'N', 'O', 'S'])
print(test_result)

---
---

## 8️⃣ **def one_of_k_encoding_unk():**

In [ ]:
def one_of_k_encoding_unk(x, allowable_set):
    """
    One-hot encode input x based on allowable_set.
    If x is not in allowable_set, it is mapped to the last element.

    Parameters:
    - x: input value (e.g., atom symbol or degree)
    - allowable_set: list of allowed values

    Returns:
    - List of 0/1 values with 1 at the matched position (or last if unknown)
    """
    if x not in allowable_set:
        x = allowable_set[-1]
    return [int(x == s) for s in allowable_set]

### 🔍 Test Time!

In [ ]:
print(one_of_k_encoding('N', ['C', 'N', 'O', 'S']))
print(one_of_k_encoding_unk('N', ['C', 'N', 'O', 'S']))
print(one_of_k_encoding_unk('N', ['C', 'n', 'O', 'S']))
print(one_of_k_encoding_unk('Xe', ['C', 'N', 'O', 'Unknown']))

---

### 🔍 Test Time again!

In [ ]:
# نسخه اصلی: خروجی True/False
def one_of_k_encoding_original(x, allowable_set):
    return list(map(lambda s: x == s, allowable_set))

# نسخه بهبودیافته: خروجی 0/1
def one_of_k_encoding_int(x, allowable_set):
    return [int(x == s) for s in allowable_set]

# تست با مقدار مشخص
x = 'N'
allowable_set = ['C', 'N', 'O', 'S']

print("Original version:", one_of_k_encoding_original(x, allowable_set))
print("Improved version:", one_of_k_encoding_int(x, allowable_set))

---
---

## 9️⃣ **def atom_features():**

In [ ]:
def atom_features(atom):

#Feature No.1: Atom symbol (element) one-hot encoded. | عنصر | unk برای سالم اجرا شدن عناصر غیرمتعارف
    symbol_set = ['C', 'N', 'O', 'S', 'F', 'Si', 'P', 'Cl', 'Br', 'Mg', 'Na', 'Ca', 'Fe', 'As', 'Al', 'I', 'B', 'V', 'K',
                  'Tl', 'Yb', 'Sb', 'Sn', 'Ag', 'Pd', 'Co', 'Se', 'Ti', 'Zn', 'H', 'Li', 'Ge', 'Cu', 'Au', 'Ni', 'Cd',
                  'In', 'Mn', 'Zr', 'Cr', 'Pt', 'Hg', 'Pb', 'Unknown'] # 43 + 1Unknown = 44
    # print("Length of symbol_set:", len(symbol_set))
    features = one_of_k_encoding_unk(atom.GetSymbol(), symbol_set)

#Feature No.2: Atom degree one-hot encoded. | درجه
    degree_set = list(range(11))
    features += one_of_k_encoding(atom.GetDegree(), degree_set) # 11

#Feature No.3: Total number of Hs one-hot encoded. | تعداد هیدروژن‌ها
    features += one_of_k_encoding_unk(atom.GetTotalNumHs(), degree_set) # 11

#Feature No.4: Implicit valence one-hot encoded. | (تخمین توسط RDKit) ظرفیت ضمنی
    features += one_of_k_encoding_unk(atom.GetImplicitValence(), degree_set) #11

#Feature No.5: Aromaticity (bool as float). | آروماتیسیته
    features += [float(atom.GetIsAromatic())] # 1

# خروجی یک آرایه نامپای از تمامی فیچرهای فوق
    return np.array(features) # 44+11+11+11+1=78

### 🔍 Test Time!

In [ ]:
mol = Chem.MolFromSmiles('Cc1cc2cc3c(C)cc(=O)oc3c(C)c2o1')  # اولین اسمایلز فایلم
atom = mol.GetAtomWithIdx(0)     # اولین اتم (کربن)
features = atom_features(atom)
print(features)
print("Vector length:", len(features))

---
---

### 🔍 Test Time again!

### **Just for a test!**

In [ ]:
# Load a random drug from drug_smiles.csv and analyze it fully
# === Load drug_smiles.csv and pick one drug randomly ===
filename = folder + 'molecular_features.csv'  # Make sure this file is uploaded in your Colab session

with open(filename, 'r') as f:
    reader = csv.reader(f)
    header = next(reader)
    rows = list(reader)

# Choose one drug randomly
random_drug = random.choice(rows)
drug_name = random_drug[0]
smiles = random_drug[1]

print("=== Drug Info ===")
print(f"DrugBankID (name) : {random_drug[0]}")
print(f"CanonicalSMILES (smile)  : {random_drug[1]}")

# === Convert SMILES to RDKit Mol object ===
mol = Chem.MolFromSmiles(smiles)
print(f"\nMol Object: {mol}")

# === Atoms ===
print("\n=== List of Atoms ===")
for atom in mol.GetAtoms():
    print(f"Atom Index = {atom.GetIdx()}, Symbol = {atom.GetSymbol()}, Degree = {atom.GetDegree()}")

# === Bonds ===
print("\n=== List of Bonds ===")
for bond in mol.GetBonds():
    a1 = bond.GetBeginAtomIdx()
    a2 = bond.GetEndAtomIdx()
    btype = bond.GetBondType()
    print(f"Bond: {a1} - {a2}, Type = {btype}")

# === Make molecular graph and visualize ===
edges = [(bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()) for bond in mol.GetBonds()]
graph = nx.Graph(edges)
labels = {atom.GetIdx(): atom.GetSymbol() for atom in mol.GetAtoms()}

plt.figure(figsize=(6, 6))
pos = nx.spring_layout(graph)
nx.draw(graph, pos, with_labels=True, labels=labels, node_color='lightgreen', node_size=800, font_size=10)
plt.title(f"Molecular Graph of {drug_name}")
plt.show()

---
---

## 🔟 **def smile_to_graph():**

In [ ]:
def smile_to_graph(smile):
    """
    Converts a SMILES string to a molecular graph.

    Parameters:
        smile (str): SMILES string representation of a molecule.

    Returns:
        c_size (int): Number of atoms
        features (list of np.array): Normalized atom feature vectors
        edge_index (np.array): Array of edges (source, target)
        g (nx.DiGraph): Directed molecular graph
    """
    mol = Chem.MolFromSmiles(smile)
    if mol is None:
        raise ValueError(f"Invalid SMILES string: {smile}")

    c_size = mol.GetNumAtoms()

    features = []
    for atom in mol.GetAtoms():
        feature = atom_features(atom)
        features.append(feature / np.sum(feature))  # normalize safely

    edges = []
    for bond in mol.GetBonds():
        edges.append([bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()])

    g = nx.Graph(edges).to_directed()

    edge_index = np.array(list(g.edges))  # for GNN use

    return c_size, features, edge_index, g

### 🔍 Test Time!

In [ ]:
# تست خروجی تابع فوق
def print_section(title, content):
    line = "=" * 40
    print(f"\n{line}\n{title}\n{line}")
    print(content)

# Test the smile_to_graph function with a simple SMILES string
smiles = "CC(C)(C)NCC(O)COc1cccc2c1CCC(=O)N2"  # DB00521
test_smile = smiles

# Run the function
c_size, features, edge_index, g = smile_to_graph(test_smile)

# Print results
print_section("Number of Atoms (c_size)", c_size)
print_section("Normalized Atom Feature Vectors (features)", features)
print_section("List of Edges (edge_index)", edge_index)
print_section("Directed Molecular Graph (g)", g)

# === Visualization of the molecular graph ===
mol = Chem.MolFromSmiles(smiles)
edges = [(bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()) for bond in mol.GetBonds()]
graph = nx.Graph(edges)
labels = {atom.GetIdx(): atom.GetSymbol() for atom in mol.GetAtoms()}

plt.figure(figsize=(4, 4))
pos = nx.spring_layout(graph)
nx.draw(graph, pos, with_labels=True, labels=labels,
        node_color='lightblue', node_size=800, font_size=10, edge_color='gray')
plt.title("Molecular Graph of DB00521")
plt.axis('off')
plt.show()

---
---

## 1️⃣1️⃣ **def load_drug_smile():**

In [ ]:
def load_drug_smile():
    path = folder + "molecular_features.csv"
    if not os.path.exists(path):
        raise FileNotFoundError(f"{path} not found. Please upload it to your Colab environment.")

    with open(path, 'r', encoding='utf-8') as f:
        reader = csv.reader(f)
        next(reader, None)

        drug_dict = {}
        drug_smile = []

        for item in reader:
            name = item[0]
            smile = item[1]

            if name not in drug_dict:
                drug_dict[name] = len(drug_dict)
            drug_smile.append(smile)

    smile_graph = {}
    for smile in tqdm(drug_smile, desc="Processing SMILES"):
        if smile not in smile_graph:
            smile_graph[smile] = smile_to_graph(smile)

    return drug_dict, drug_smile, smile_graph

### 🔍 Test Time!

In [ ]:
# =======================================================
# Test the load_drug_smile() function and inspect outputs
# =======================================================

drug_dict, drug_smile, smile_graph = load_drug_smile()

print("\n" + "="*70)
print("✅ DATA SUMMARY")
print("="*70)
print(f"• Number of drugs (in drug_dict)  : {len(drug_dict):,}")
print(f"• Number of SMILES (drug_smile) : {len(drug_smile):,}")
print(f"• Number of graphs (smile_graph): {len(smile_graph):,}")

# -------------------------------------------------------
# Preview first N items for each structure
# -------------------------------------------------------
PREVIEW_COUNT = 3

print("\n" + "="*70)
print("🔍 PREVIEW: drug_dict (ID → SMILES)")
print("="*70)

for i, (drug_id, smile) in enumerate(drug_dict.items()):
    print(f"{i+1:>2}) {drug_id} → {smile}")
    if i + 1 == PREVIEW_COUNT:
        break


print("\n" + "="*70)
print("🔍 PREVIEW: drug_smile (list of SMILES)")
print("="*70)

for i, smile in enumerate(drug_smile[:PREVIEW_COUNT]):
    print(f"{i+1:>2}) {smile}")


print("\n" + "="*70)
print("🔍 PREVIEW: smile_graph (SMILES → Graph structure)")
print("="*70)

for i, (smile, graph_data) in enumerate(smile_graph.items()):
    print(f"{i+1:>2}) SMILES: {smile}")
    c_size, features, edge_index, g = graph_data
    print(f"     • Num atoms        : {c_size}")
    print(f"     • Feature dim      : {len(features[0]) if len(features)>0 else 0}")
    print(f"     • Num edges        : {len(edge_index)}")
    print(f"     • Graph object     : {type(g)}")
    if i + 1 == PREVIEW_COUNT:
        break


# -------------------------------------------------------
# Example detailed output for a single SMILES
# -------------------------------------------------------
print("\n" + "="*70)
print("📌 Example: detailed graph info for the FIRST SMILES")
print("="*70)

if len(drug_smile) > 0:
    example_smile = drug_smile[0]
    print("Example SMILES:", example_smile)
    c_size, features, edge_index, g = smile_graph[example_smile]
    print("  - Number of atoms                :", c_size)
    print("  - Atom feature vector length     :", len(features[0]))
    print("  - Number of edges (bonds)        :", len(edge_index))
else:
    print("⛔ drug_smile is EMPTY!")

---
---

### 📥 **2) `DrugGraphDataset` CLASS**

In [ ]:
# تبدیل smile_graph به لیست گراف‌های torch_geometric
def convert_smile_graph_to_data_list(smile_graph, drug_dict, drug_smile):
    data_list = []
    for i, smile in enumerate(drug_smile):
        try:
            c_size, features, edge_index, _ = smile_graph[smile]
            x = torch.tensor(features, dtype=torch.float)
            edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()

            data = Data(x=x, edge_index=edge_index)
            data.smile = smile
            data.drug_name = list(drug_dict.keys())[list(drug_dict.values()).index(i)]
            data_list.append(data)
        except Exception as e:
            print(f"❌ Error processing {smile[:10]}...: {e}")
            continue
    return data_list

# استفاده:
drug_dict, drug_smile, smile_graph = load_drug_smile()
data_list = convert_smile_graph_to_data_list(smile_graph, drug_dict, drug_smile)
print(f"✅ تعداد گراف آماده برای GNN: {len(data_list)}")

In [ ]:
class DrugGraphDataset(InMemoryDataset):
    def __init__(self, data_list, root='.'):
        self.data_list = data_list
        super().__init__(root)
        self.data, self.slices = self.collate(data_list)

    def process(self):
        pass

    @property
    def raw_file_names(self): return []
    @property
    def processed_file_names(self): return ['not_used.pt']

dataset = DrugGraphDataset(data_list)
torch.save(dataset, folder + 'gnn_ready_drug_graphs.pt')
print("📦 ذخیره شد: gnn_ready_drug_graphs.pt")

### 🔍 Test Time!

In [ ]:
dataset_loaded = torch.load(folder + 'gnn_ready_drug_graphs.pt', weights_only=False)

print(f"📦 تعداد گراف‌ها: {len(dataset_loaded)}")

# نمایش اولین گراف به عنوان نمونه
print("\n🔍 گراف اول:")
print(dataset_loaded[0])

In [ ]:
# بارگذاری دیتاست ذخیره‌شده
dataset_loaded = torch.load(folder + 'gnn_ready_drug_graphs.pt', weights_only=False)

print(f"📦 تعداد گراف‌ها: {len(dataset_loaded)}")

# نمایش اطلاعات اولین گراف
data = dataset_loaded[0]

print("\n🔍 گراف اول:")
print(f"🧪 دارو (نام): {getattr(data, 'drug_name', 'نامشخص')}")
print(f"🧬 SMILES: {getattr(data, 'smile', 'نامشخص')}")

print("\n📌 ویژگی گره‌ها (x):")
print(f"x شامل ویژگی‌های اتم‌ها در گراف مولکولی است — مثلاً نوع اتم، ظرفیت، تعداد هیدروژن و غیره")
print(f"شکل (shape): {data.x.shape}")
print(data.x)

print("\n📌 ساختار گراف (edge_index):")
print("edge_index شامل اتصال بین اتم‌هاست (لیستی از یال‌ها: از-به)")
print(f"شکل (shape): {data.edge_index.shape}")
print(data.edge_index)

# اگر خواستی سایر ویژگی‌ها مثل edge_attr یا هر چی اضافه شده رو هم بررسی کنی:
# optional_fields = ['edge_attr', 'pos', 'edge_weight']
# for field in optional_fields:
#     if hasattr(data, field):
#         print(f"\n📌 {field}:")
#         print(getattr(data, field))

---
---
---

# 📌 **GNNs Part!**

هدف اینه که روی هر دارو، که یک گرافه، یک شبکه عصبی گرافی اجرا کنیم و در نهایت برای کل گرافِ دارو یک امبدینگ وکتور، مثلاً با طول 128، بگیریم

# 2️⃣🧪 **Drug GCN**

### Key Improvements in the DrugGCN

- The model is **more complex** and utilizes **hierarchical feature extraction**.
- It incorporates **regularization (Dropout)** to prevent overfitting.
- **Result:** *Embeddings are richer, more robust, and more resistant to overfitting.*

✅ **Step 1:** Load the graphs

In [ ]:
dataset_loaded = torch.load(folder + 'gnn_ready_drug_graphs.pt', weights_only=False)
print(f"📦 تعداد داروها (گراف‌ها): {len(dataset_loaded)}")

In [ ]:
type(dataset_loaded), len(dataset_loaded)

✅ **Step 2:** Create the DataLoader (in batches)

In [ ]:
# 📥 استخراج embedding داروها
batch_size = 64
loader = DataLoader(dataset_loaded, batch_size=batch_size, shuffle=False)

✅ **Step 3:** Define a **`DrugGCN`** model

In [ ]:
class DrugGCN(nn.Module):
    def __init__(self, num_features_xd=78, hidden_channels=256, out_channels=256, dropout=0.5): #num_features_xd=78, hidden_channels=128, out_channels=128 | 1024
        super(DrugGCN, self).__init__()

        self.conv1 = GCNConv(num_features_xd, hidden_channels) # لایه 1 GCN
        self.conv2 = GCNConv(hidden_channels, hidden_channels * 2)  # لایه 2 GCN
        self.conv3 = GCNConv(hidden_channels * 2, hidden_channels * 4)  # لایه 3 GCN

        # چند لایه‌ی fully-connected (Linear) اضافه شده
        self.fc1 = nn.Linear(hidden_channels * 4, 1024)
        self.fc2 = nn.Linear(1024, out_channels)

        # بین لایه‌ها BatchNorm و Dropout هم اضافه شده
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, edge_index, batch):
        # x = F.relu(self.conv1(x, edge_index))
        x = self.conv1(x, edge_index)
        x = self.relu(x)
        # x = F.relu(self.conv2(x, edge_index))
        x = self.conv2(x, edge_index)
        x = self.relu(x)
        # x = F.relu(self.conv3(x, edge_index))
        x = self.conv3(x, edge_index)
        x = self.relu(x)

        # به جای global_mean_pool، از global_max_pool استفاده کرده
        # x = global_max_pool(x, batch)
        x = global_max_pool(x, batch)  # pool per-graph → [batch_size, hidden*4]

        # x = F.relu(self.fc1(x))
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)  # Final embedding # x = self.fc2(x)

        return x  # [batch_size, out_channels]

چک مجدد

GCNConv layers:    78 → 256 → 512 → 1024
FC layers:         1024 → 512 → 256

self.fc1 = nn.Linear(hidden_channels * 4, hidden_channels * 2)   # 1024 → 512
self.fc2 = nn.Linear(hidden_channels * 2, out_channels)          # 512 → 256

✅ **Step 4:** Run the model and obtain the embeddings

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = DrugGCN(num_features_xd=78).to(device)
model.eval()

all_embeddings = []

# بخش استخراج و ذخیره‌ی embeddings
for batch in loader:
    batch = batch.to(device)
    with torch.no_grad():
        emb = model(batch.x, batch.edge_index, batch.batch)
        all_embeddings.append(emb.cpu())

drug_embeddings = torch.cat(all_embeddings, dim=0) # بخش استخراج و ذخیره‌ی embeddings
print("✅ drug_embeddings shape:", drug_embeddings.shape)  # [num_drugs, 128]

***

✅ **Step 4.1:** PCA (Reduce to 200 by PCA)

In [ ]:
# --- Reduce embeddings to 200 dims by PCA ---
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import joblib  # برای ذخیره اسکیلر و PCA

# 1) به numpy
X = drug_embeddings.numpy() if isinstance(drug_embeddings, torch.Tensor) else np.asarray(drug_embeddings)
print("raw embeddings:", X.shape)  # [num_drugs, 1024]

# 2) نرمال‌سازی (میانگین‌گیری و مقیاس‌بندی) قبل از PCA
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 3) PCA → 200 بُعد
pca = PCA(n_components=200, svd_solver='auto', random_state=42)
X_200 = pca.fit_transform(X_scaled)
print("PCA(200) shape:", X_200.shape)  # [num_drugs, 200]

# 4) جایگزین کردن امبدینگ‌ها با نسخه 200بُعدی (و تبدیل به float32 برای PyTorch/ذخیره)
drug_embeddings = torch.from_numpy(X_200.astype(np.float32))

# 5) (اختیاری) ذخیره اسکیلر/PCA برای استفاده در inference تا سازگاری حفظ شود
joblib.dump(scaler, "pca_scaler.pkl")
joblib.dump(pca, "pca_200.pkl")

***

✅ **Step 5:** Save the embeddings

In [ ]:
# ذخیره‌ی 200بعدی به‌صورت .npy
np.save(folder + "drug_embeddings_gcnnet_200.npy", drug_embeddings.numpy())

In [ ]:
# نسخه‌ی متنی که train.py می‌خوانَد
np.savetxt("PCA.txt", drug_embeddings.numpy(), fmt="%.6f")

✅ **Step 6:** Command to display embeddings in Colab

In [ ]:
# بارگذاری embeddingهایی که ذخیره کردی
embeddings = np.load(folder + "drug_embeddings_gcnnet_200.npy")  # ⬅️ تغییر نام فایل

# تبدیل به DataFrame
df = pd.DataFrame(embeddings)
df.columns = [f"dim_{i}" for i in range(df.shape[1])]  # dim_0 تا dim_199
df.index.name = "Drug_Index"

# نمایش ۵ سطر اول
print("✅ نمونه‌ای از embedding داروها (200 بعدی):")
df.head()

---
---